In [ ]:
import os, subprocess, sys, time
os.chdir('/kaggle/working')
print('ETAPE1: clone + deps')
r = subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git'],capture_output=True,text=True)
print('clone rc=', r.returncode, r.stderr[-300:] if r.returncode else '')
r = subprocess.run([sys.executable,'-m','pip','install','-q','-r','ComfyUI/requirements.txt'],capture_output=True,text=True)
print('req rc=', r.returncode, r.stderr[-300:] if r.returncode else '')
r = subprocess.run([sys.executable,'-m','pip','install','-q','aiohttp'],capture_output=True,text=True)
print('aiohttp rc=', r.returncode)
print('ETAPE1 FINIE')

In [ ]:
print('ETAPE2: modèles depuis dataset Kaggle monté, aucun téléchargement externe')
import os, pathlib, shutil, json
input_root=pathlib.Path('/kaggle/input')
if not input_root.exists(): raise RuntimeError('Kaggle /kaggle/input absent')
matches=list(input_root.rglob('ltx-video-2b-v0.9.5.safetensors'))
t5=list(input_root.rglob('t5xxl_fp8_e4m3fn.safetensors'))
if not matches: raise RuntimeError('ltx-video-2b-v0.9.5.safetensors absent des datasets Kaggle attaches')
if not t5: raise RuntimeError('t5xxl_fp8_e4m3fn.safetensors absent des datasets Kaggle attaches')
root=pathlib.Path('/kaggle/working/ComfyUI/models')
(root/'checkpoints').mkdir(parents=True,exist_ok=True); (root/'text_encoders').mkdir(parents=True,exist_ok=True)
shutil.copy2(matches[0],root/'checkpoints/ltx-video-2b-v0.9.5.safetensors')
shutil.copy2(t5[0],root/'text_encoders/t5xxl_fp8_e4m3fn.safetensors')
print('dataset models:',matches[0],t5[0])
print('ETAPE2 FINIE')


In [ ]:
print('ETAPE3: workflows video embarques')
import json, pathlib
WORKFLOWS={"LTXV_image_to_video.json": {"id": "cfe20fa9-dc56-41eb-990a-7c6f31a90284", "revision": 0, "last_node_id": 235, "last_link_id": 415, "nodes": [{"id": 38, "type": "CLIPLoader", "pos": [60, 190], "size": [315, 106], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "slot_index": 0, "links": [74, 75]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "CLIPLoader", "models": [{"name": "t5xxl_fp16.safetensors", "url": "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors", "directory": "text_encoders"}]}, "widgets_values": ["t5xxl_fp8_e4m3fn.safetensors", "ltxv", "default"]}, {"id": 69, "type": "LTXVConditioning", "pos": [640, 800], "size": [210, 80], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "positive", "type": "CONDITIONING", "link": 183}, {"name": "negative", "type": "CONDITIONING", "link": 184}], "outputs": [{"name": "positive", "type": "CONDITIONING", "slot_index": 0, "links": [166]}, {"name": "negative", "type": "CONDITIONING", "slot_index": 1, "links": [167]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "LTXVConditioning"}, "widgets_values": [25]}, {"id": 76, "type": "Note", "pos": [420, -130], "size": [410, 90], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [], "properties": {}, "widgets_values": ["This model needs long descriptive prompts, if the prompt is too short the quality will suffer greatly."], "color": "#222", "bgcolor": "#000"}, {"id": 7, "type": "CLIPTextEncode", "pos": [420, 320], "size": [425.2799987792969, 180.61000061035156], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 75}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "slot_index": 0, "links": [188]}], "title": "CLIP Text Encode (Negative Prompt)", "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["low quality, worst quality, deformed, distorted, disfigured, motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly"], "color": "#223", "bgcolor": "#335"}, {"id": 71, "type": "LTXVScheduler", "pos": [910, 180], "size": [350, 154], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "latent", "shape": 7, "type": "LATENT", "link": 185}], "outputs": [{"name": "SIGMAS", "type": "SIGMAS", "slot_index": 0, "links": [182]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "LTXVScheduler"}, "widgets_values": [30, 2.05, 0.95, true, 0.1]}, {"id": 73, "type": "KSamplerSelect", "pos": [910, 70], "size": [350, 60], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [{"name": "SAMPLER", "type": "SAMPLER", "links": [172]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "KSamplerSelect"}, "widgets_values": ["euler"]}, {"id": 72, "type": "SamplerCustom", "pos": [1280, 70], "size": [355.20001220703125, 442], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 181}, {"name": "positive", "type": "CONDITIONING", "link": 166}, {"name": "negative", "type": "CONDITIONING", "link": 167}, {"name": "sampler", "type": "SAMPLER", "link": 172}, {"name": "sigmas", "type": "SIGMAS", "link": 182}, {"name": "latent_image", "type": "LATENT", "link": 186}], "outputs": [{"name": "output", "type": "LATENT", "slot_index": 0, "links": [171]}, {"name": "denoised_output", "type": "LATENT", "links": null}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "SamplerCustom"}, "widgets_values": [true, 712395695050634, "randomize", 3]}, {"id": 81, "type": "SaveVideo", "pos": [1680, 10], "size": [720, 890], "flags": {}, "order": 14, "mode": 0, "inputs": [{"name": "video", "type": "VIDEO", "link": 191}], "outputs": [], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "SaveVideo"}, "widgets_values": ["video/ComfyUI", "auto", "auto"]}, {"id": 44, "type": "CheckpointLoaderSimple", "pos": [60, 50], "size": [315, 98], "flags": {}, "order": 3, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [181]}, {"name": "CLIP", "type": "CLIP", "links": null}, {"name": "VAE", "type": "VAE", "slot_index": 2, "links": [87, 189]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "CheckpointLoaderSimple", "models": [{"name": "ltx-video-2b-v0.9.5.safetensors", "url": "https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors", "directory": "checkpoints"}]}, "widgets_values": ["ltx-video-2b-v0.9.5.safetensors"]}, {"id": 80, "type": "CreateVideo", "pos": [1410, 620], "size": [220, 80], "flags": {}, "order": 13, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 192}, {"name": "audio", "shape": 7, "type": "AUDIO", "link": null}], "outputs": [{"name": "VIDEO", "type": "VIDEO", "links": [191]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "CreateVideo"}, "widgets_values": [24]}, {"id": 8, "type": "VAEDecode", "pos": [1120, 620], "size": [210, 46], "flags": {}, "order": 12, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 171}, {"name": "vae", "type": "VAE", "link": 87}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "slot_index": 0, "links": [192]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "VAEDecode"}, "widgets_values": []}, {"id": 6, "type": "CLIPTextEncode", "pos": [420, 90], "size": [422.8500061035156, 164.30999755859375], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 74}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "slot_index": 0, "links": [187]}], "title": "CLIP Text Encode (Positive Prompt)", "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["A red fox moving gracefully, its russet coat vibrant against the white landscape, leaving perfect star-shaped prints behind as steam rises from its breath in the crisp winter air. The scene is wrapped in snow-muffled silence, broken only by the gentle murmur of water still flowing beneath the ice."], "color": "#232", "bgcolor": "#353"}, {"id": 78, "type": "LoadImage", "pos": [70, 400], "size": [310, 320], "flags": {}, "order": 4, "mode": 0, "inputs": [], "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [190]}, {"name": "MASK", "type": "MASK", "links": null}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "LoadImage"}, "widgets_values": ["ltxv_image_to_video_input_image.jpg", "image"]}, {"id": 77, "type": "LTXVImgToVideo", "pos": [420, 670], "size": [210, 214], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "positive", "type": "CONDITIONING", "link": 187}, {"name": "negative", "type": "CONDITIONING", "link": 188}, {"name": "vae", "type": "VAE", "link": 189}, {"name": "image", "type": "IMAGE", "link": 190}], "outputs": [{"name": "positive", "type": "CONDITIONING", "slot_index": 0, "links": [183]}, {"name": "negative", "type": "CONDITIONING", "slot_index": 1, "links": [184]}, {"name": "latent", "type": "LATENT", "slot_index": 2, "links": [185, 186]}], "properties": {"cnr_id": "comfy-core", "ver": "0.3.65", "Node name for S&R": "LTXVImgToVideo"}, "widgets_values": [768, 512, 97, 1, 0.15]}, {"id": 79, "type": "MarkdownNote", "pos": [-390, 20], "size": [430, 280], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [], "properties": {}, "widgets_values": ["\n## Model links\n\n**checkpoints**\n\n- [ltx-video-2b-v0.9.5.safetensors](https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors) (5.91 GB)\n\n**text_encoders**\n\n- [t5xxl_fp16.safetensors](https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors) (9.12 GB)\n\n\nModel Storage Location\n\n```\n📂 ComfyUI/\n├── 📂 models/\n│   ├── 📂 checkpoints/\n│   │      └── ltx-video-2b-v0.9.5.safetensors\n│   └── 📂 text_encoders/\n│          └── t5xxl_fp16.safetensors\n```\n"], "color": "#222", "bgcolor": "#000"}], "links": [[74, 38, 0, 6, 0, "CLIP"], [75, 38, 0, 7, 0, "CLIP"], [87, 44, 2, 8, 1, "VAE"], [166, 69, 0, 72, 1, "CONDITIONING"], [167, 69, 1, 72, 2, "CONDITIONING"], [171, 72, 0, 8, 0, "LATENT"], [172, 73, 0, 72, 3, "SAMPLER"], [181, 44, 0, 72, 0, "MODEL"], [182, 71, 0, 72, 4, "SIGMAS"], [183, 77, 0, 69, 0, "CONDITIONING"], [184, 77, 1, 69, 1, "CONDITIONING"], [185, 77, 2, 71, 0, "LATENT"], [186, 77, 2, 72, 5, "LATENT"], [187, 6, 0, 77, 0, "CONDITIONING"], [188, 7, 0, 77, 1, "CONDITIONING"], [189, 44, 2, 77, 2, "VAE"], [190, 78, 0, 77, 3, "IMAGE"], [191, 80, 0, 81, 0, "VIDEO"], [192, 8, 0, 80, 0, "IMAGE"]], "groups": [{"id": 5, "title": "Step1- Load models", "bounding": [50, -20, 340, 330], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 6, "title": "Step3 - Prompt", "bounding": [410, -20, 445.280029296875, 604.2099609375], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 7, "title": "Video size", "bounding": [410, 600, 450, 300], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 8, "title": "Custom sampler", "bounding": [880, -20, 770, 580], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 9, "title": "Step2 - Upload image", "bounding": [50, 330, 340, 410], "color": "#3f789e", "font_size": 24, "flags": {}}], "config": {}, "extra": {"ds": {"scale": 0.26910083463845624, "offset": [2457.4033298244667, 1666.174444471151]}, "frontendVersion": "1.27.10", "VHS_latentpreview": false, "VHS_latentpreviewrate": 0, "VHS_MetadataImage": true, "VHS_KeepIntermediate": true}, "version": 0.4}, "LTXV_text_to_video.json": {"id": "44b59e39-769c-43b9-a97c-651f369294c8", "revision": 0, "last_node_id": 79, "last_link_id": 184, "nodes": [{"id": 38, "type": "CLIPLoader", "pos": [60, 190], "size": [315, 106], "flags": {}, "order": 0, "mode": 0, "inputs": [], "outputs": [{"name": "CLIP", "type": "CLIP", "slot_index": 0, "links": [74, 75]}], "properties": {"Node name for S&R": "CLIPLoader", "models": [{"name": "t5xxl_fp16.safetensors", "url": "https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors?download=true", "directory": "text_encoders"}]}, "widgets_values": ["t5xxl_fp8_e4m3fn.safetensors", "ltxv", "default"]}, {"id": 44, "type": "CheckpointLoaderSimple", "pos": [60, 40], "size": [320, 100], "flags": {}, "order": 1, "mode": 0, "inputs": [], "outputs": [{"name": "MODEL", "type": "MODEL", "slot_index": 0, "links": [181]}, {"name": "CLIP", "type": "CLIP", "links": null}, {"name": "VAE", "type": "VAE", "slot_index": 2, "links": [87]}], "properties": {"Node name for S&R": "CheckpointLoaderSimple", "models": [{"name": "ltx-video-2b-v0.9.safetensors", "url": "https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.safetensors?download=true", "directory": "checkpoints"}]}, "widgets_values": ["ltx-video-2b-v0.9.safetensors"]}, {"id": 7, "type": "CLIPTextEncode", "pos": [420, 380], "size": [425.2799987792969, 180.61000061035156], "flags": {}, "order": 7, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 75}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "slot_index": 0, "links": [170]}], "title": "CLIP Text Encode (Negative Prompt)", "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["low quality, worst quality, deformed, distorted, disfigured, motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly"], "color": "#223", "bgcolor": "#335"}, {"id": 6, "type": "CLIPTextEncode", "pos": [420, 180], "size": [422.8500061035156, 164.30999755859375], "flags": {}, "order": 6, "mode": 0, "inputs": [{"name": "clip", "type": "CLIP", "link": 74}], "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "slot_index": 0, "links": [169]}], "title": "CLIP Text Encode (Positive Prompt)", "properties": {"Node name for S&R": "CLIPTextEncode"}, "widgets_values": ["A woman with long brown hair and light skin smiles at another woman with long blonde hair. The woman with brown hair wears a black jacket and has a small, barely noticeable mole on her right cheek. The camera angle is a close-up, focused on the woman with brown hair's face. The lighting is warm and natural, likely from the setting sun, casting a soft glow on the scene. The scene appears to be real-life footage."], "color": "#232", "bgcolor": "#353"}, {"id": 76, "type": "Note", "pos": [420, 40], "size": [420, 90], "flags": {}, "order": 2, "mode": 0, "inputs": [], "outputs": [], "properties": {}, "widgets_values": ["This model needs long descriptive prompts, if the prompt is too short the quality will suffer greatly."], "color": "#222", "bgcolor": "#000"}, {"id": 70, "type": "EmptyLTXVLatentVideo", "pos": [60, 390], "size": [315, 130], "flags": {}, "order": 3, "mode": 0, "inputs": [], "outputs": [{"name": "LATENT", "type": "LATENT", "slot_index": 0, "links": [168, 175]}], "properties": {"Node name for S&R": "EmptyLTXVLatentVideo"}, "widgets_values": [768, 512, 97, 1]}, {"id": 72, "type": "SamplerCustom", "pos": [890, 340], "size": [330, 230], "flags": {}, "order": 10, "mode": 0, "inputs": [{"name": "model", "type": "MODEL", "link": 181}, {"name": "positive", "type": "CONDITIONING", "link": 166}, {"name": "negative", "type": "CONDITIONING", "link": 167}, {"name": "sampler", "type": "SAMPLER", "link": 172}, {"name": "sigmas", "type": "SIGMAS", "link": 182}, {"name": "latent_image", "type": "LATENT", "link": 175}], "outputs": [{"name": "output", "type": "LATENT", "slot_index": 0, "links": [171]}, {"name": "denoised_output", "type": "LATENT", "links": null}], "properties": {"Node name for S&R": "SamplerCustom"}, "widgets_values": [true, 497797676867141, "randomize", 3]}, {"id": 73, "type": "KSamplerSelect", "pos": [890, 240], "size": [315, 58], "flags": {}, "order": 4, "mode": 0, "inputs": [], "outputs": [{"name": "SAMPLER", "type": "SAMPLER", "links": [172]}], "properties": {"Node name for S&R": "KSamplerSelect"}, "widgets_values": ["euler"]}, {"id": 71, "type": "LTXVScheduler", "pos": [890, 40], "size": [315, 154], "flags": {}, "order": 8, "mode": 0, "inputs": [{"name": "latent", "shape": 7, "type": "LATENT", "link": 168}], "outputs": [{"name": "SIGMAS", "type": "SIGMAS", "slot_index": 0, "links": [182]}], "properties": {"Node name for S&R": "LTXVScheduler"}, "widgets_values": [30, 2.05, 0.95, true, 0.1]}, {"id": 69, "type": "LTXVConditioning", "pos": [630, 620], "size": [223.8699951171875, 78], "flags": {}, "order": 9, "mode": 0, "inputs": [{"name": "positive", "type": "CONDITIONING", "link": 169}, {"name": "negative", "type": "CONDITIONING", "link": 170}], "outputs": [{"name": "positive", "type": "CONDITIONING", "slot_index": 0, "links": [166]}, {"name": "negative", "type": "CONDITIONING", "slot_index": 1, "links": [167]}], "properties": {"Node name for S&R": "LTXVConditioning"}, "widgets_values": [25]}, {"id": 8, "type": "VAEDecode", "pos": [1250, 0], "size": [210, 46], "flags": {}, "order": 11, "mode": 0, "inputs": [{"name": "samples", "type": "LATENT", "link": 171}, {"name": "vae", "type": "VAE", "link": 87}], "outputs": [{"name": "IMAGE", "type": "IMAGE", "slot_index": 0, "links": [183]}], "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": []}, {"id": 78, "type": "CreateVideo", "pos": [1250, 90], "size": [220, 80], "flags": {}, "order": 12, "mode": 0, "inputs": [{"name": "images", "type": "IMAGE", "link": 183}, {"name": "audio", "shape": 7, "type": "AUDIO", "link": null}], "outputs": [{"name": "VIDEO", "type": "VIDEO", "links": [184]}], "properties": {"Node name for S&R": "CreateVideo"}, "widgets_values": [24]}, {"id": 79, "type": "SaveVideo", "pos": [1250, 220], "size": [450, 370], "flags": {}, "order": 13, "mode": 0, "inputs": [{"name": "video", "type": "VIDEO", "link": 184}], "outputs": [], "properties": {"Node name for S&R": "SaveVideo"}, "widgets_values": ["video/ComfyUI", "auto", "auto"]}, {"id": 77, "type": "MarkdownNote", "pos": [-190, 0], "size": [225, 88], "flags": {}, "order": 5, "mode": 0, "inputs": [], "outputs": [], "properties": {}, "widgets_values": ["🛈 [Learn more about this workflow](https://comfyanonymous.github.io/ComfyUI_examples/ltxv/)"], "color": "#222", "bgcolor": "#000"}], "links": [[74, 38, 0, 6, 0, "CLIP"], [75, 38, 0, 7, 0, "CLIP"], [87, 44, 2, 8, 1, "VAE"], [166, 69, 0, 72, 1, "CONDITIONING"], [167, 69, 1, 72, 2, "CONDITIONING"], [168, 70, 0, 71, 0, "LATENT"], [169, 6, 0, 69, 0, "CONDITIONING"], [170, 7, 0, 69, 1, "CONDITIONING"], [171, 72, 0, 8, 0, "LATENT"], [172, 73, 0, 72, 3, "SAMPLER"], [175, 70, 0, 72, 5, "LATENT"], [181, 44, 0, 72, 0, "MODEL"], [182, 71, 0, 72, 4, "SIGMAS"], [183, 8, 0, 78, 0, "IMAGE"], [184, 78, 0, 79, 0, "VIDEO"]], "groups": [{"id": 1, "title": "Step1 - Load models", "bounding": [50, -30, 340, 339.6000061035156], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 2, "title": "Step3 - Prompt", "bounding": [410, -30, 445.280029296875, 604.2099609375], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 3, "title": "Step2 - Video size", "bounding": [50, 320, 340, 220], "color": "#3f789e", "font_size": 24, "flags": {}}, {"id": 4, "title": "Custom sampler", "bounding": [880, -30, 350, 613.5999755859375], "color": "#3f789e", "font_size": 24, "flags": {}}], "config": {}, "extra": {"ds": {"scale": 1.2666661150000043, "offset": [282.3491595411123, 142.22408251179098]}, "frontendVersion": "1.25.3"}, "version": 0.4}, "KaggleVideo.json": {"id": "587ab6f7-2224-4f1e-9a6d-18d5d2254617", "revision": 0, "last_node_id": 8, "last_link_id": 14, "nodes": [{"id": 1, "type": "AgyTextPrompt", "pos": [40, 80], "size": [380, 160], "flags": {}, "order": 0, "mode": 0, "inputs": [{"localized_name": "text", "name": "text", "type": "STRING", "widget": {"name": "text"}, "link": null}], "outputs": [{"localized_name": "text", "name": "text", "type": "STRING", "slot_index": 0, "links": [10]}], "title": "PROMPT POSITIF", "properties": {"Node name for S&R": "AgyTextPrompt"}, "widgets_values": ["a cute astronaut cat on the moon, digital art, high quality"], "widgets_values_named": {"text": "a cute astronaut cat on the moon, digital art, high quality"}}, {"id": 2, "type": "AgyTextPrompt", "pos": [40, 320], "size": [380, 160], "flags": {}, "order": 1, "mode": 0, "inputs": [{"localized_name": "text", "name": "text", "type": "STRING", "widget": {"name": "text"}, "link": null}], "outputs": [{"localized_name": "text", "name": "text", "type": "STRING", "slot_index": 0, "links": [11]}], "title": "PROMPT NEGATIF", "properties": {"Node name for S&R": "AgyTextPrompt"}, "widgets_values": ["blurry, deformed, low quality, text"], "widgets_values_named": {"text": "blurry, deformed, low quality, text"}}, {"id": 3, "type": "AgyProviderRouter", "pos": [480, 80], "size": [440, 542], "flags": {}, "order": 2, "mode": 0, "inputs": [{"localized_name": "images", "name": "images", "shape": 7, "type": "IMAGE", "link": null}, {"localized_name": "prompt", "name": "prompt", "type": "STRING", "widget": {"name": "prompt"}, "link": 10}, {"localized_name": "provider", "name": "provider", "type": "COMBO", "widget": {"name": "provider"}, "link": null}, {"localized_name": "model", "name": "model", "type": "COMBO", "widget": {"name": "model"}, "link": null}, {"localized_name": "mode", "name": "mode", "type": "COMBO", "widget": {"name": "mode"}, "link": null}, {"localized_name": "negative_prompt", "name": "negative_prompt", "shape": 7, "type": "STRING", "widget": {"name": "negative_prompt"}, "link": 11}, {"localized_name": "external_api_key", "name": "external_api_key", "shape": 7, "type": "STRING", "widget": {"name": "external_api_key"}, "link": null}, {"localized_name": "width", "name": "width", "shape": 7, "type": "INT", "widget": {"name": "width"}, "link": null}, {"localized_name": "height", "name": "height", "shape": 7, "type": "INT", "widget": {"name": "height"}, "link": null}, {"localized_name": "steps", "name": "steps", "shape": 7, "type": "INT", "widget": {"name": "steps"}, "link": null}, {"localized_name": "cfg", "name": "cfg", "shape": 7, "type": "FLOAT", "widget": {"name": "cfg"}, "link": null}, {"localized_name": "seed", "name": "seed", "shape": 7, "type": "INT", "widget": {"name": "seed"}, "link": null}, {"localized_name": "denoise", "name": "denoise", "shape": 7, "type": "FLOAT", "widget": {"name": "denoise"}, "link": null}, {"localized_name": "lora_name", "name": "lora_name", "shape": 7, "type": "STRING", "widget": {"name": "lora_name"}, "link": null}, {"localized_name": "lora_strength", "name": "lora_strength", "shape": 7, "type": "FLOAT", "widget": {"name": "lora_strength"}, "link": null}, {"localized_name": "frames", "name": "frames", "shape": 7, "type": "INT", "widget": {"name": "frames"}, "link": null}, {"localized_name": "fps", "name": "fps", "shape": 7, "type": "INT", "widget": {"name": "fps"}, "link": null}], "outputs": [{"localized_name": "text", "name": "text", "type": "STRING", "slot_index": 0, "links": [12]}, {"localized_name": "image_url", "name": "image_url", "type": "STRING", "slot_index": 1, "links": [13]}, {"localized_name": "image", "name": "image", "type": "IMAGE", "slot_index": 2, "links": [14, 15]}], "title": "KAGGLE VIDEO (LTXV distant -> frames)", "properties": {"Node name for S&R": "AgyProviderRouter"}, "widgets_values": ["", "kaggle", "RealVisXL_V5.0_fp16.safetensors", "video", "blurry, low quality, watermark, motion smear", "", 768, 512, 30, 3, -1, "randomize", 1, "", 1, 97, 25], "widgets_values_named": {"prompt": "", "provider": "kaggle", "model": "RealVisXL_V5.0_fp16.safetensors", "mode": "video", "negative_prompt": "blurry, low quality, watermark, motion smear", "external_api_key": "", "width": 768, "height": 512, "steps": 30, "cfg": 3, "seed": -1, "control_after_generate": "randomize", "denoise": 1, "lora_name": "", "lora_strength": 1, "frames": 97, "fps": 25}}, {"id": 4, "type": "PreviewImage", "pos": [980, 300], "size": [420, 380], "flags": {}, "order": 5, "mode": 0, "inputs": [{"localized_name": "images", "name": "images", "type": "IMAGE", "link": 14}], "outputs": [{"localized_name": "images", "name": "images", "type": "IMAGE", "links": null}], "properties": {"cnr_id": "comfy-core", "ver": "0.43.0", "Node name for S&R": "PreviewImage"}}, {"id": 5, "type": "PreviewAny", "pos": [980, 60], "size": [420, 180], "flags": {}, "order": 3, "mode": 0, "inputs": [{"localized_name": "source", "name": "source", "type": "*", "link": 12}], "outputs": [{"localized_name": "CHAÎNE", "name": "STRING", "type": "STRING", "links": null}], "title": "text", "properties": {"cnr_id": "comfy-core", "ver": "0.43.0", "Node name for S&R": "PreviewAny"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 6, "type": "PreviewAny", "pos": [980, 720], "size": [420, 180], "flags": {}, "order": 4, "mode": 0, "inputs": [{"localized_name": "source", "name": "source", "type": "*", "link": 13}], "outputs": [{"localized_name": "CHAÎNE", "name": "STRING", "type": "STRING", "links": null}], "title": "image_url", "properties": {"cnr_id": "comfy-core", "ver": "0.43.0", "Node name for S&R": "PreviewAny"}, "widgets_values": [], "widgets_values_named": {}}, {"id": 7, "type": "CreateVideo", "pos": [1532.4150596877867, 78.60422405876989], "size": [210, 150], "flags": {}, "order": 6, "mode": 0, "inputs": [{"localized_name": "images", "name": "images", "type": "IMAGE", "link": 15}, {"localized_name": "audio", "name": "audio", "shape": 7, "type": "AUDIO", "link": null}, {"localized_name": "fps", "name": "fps", "type": "FLOAT", "widget": {"name": "fps"}, "link": null}, {"localized_name": "bit_depth", "name": "bit_depth", "shape": 7, "type": "COMBO", "widget": {"name": "bit_depth"}, "link": null}, {"localized_name": "color_space", "name": "color_space", "shape": 7, "type": "COMBO", "widget": {"name": "color_space"}, "link": null}, {"localized_name": "codec", "name": "codec", "shape": 7, "type": "COMBO", "widget": {"name": "codec"}, "link": null}], "outputs": [{"localized_name": "Vidéo", "name": "VIDEO", "type": "VIDEO", "slot_index": 0, "links": [16]}], "title": "ASSEMBLAGE VIDEO", "properties": {"cnr_id": "comfy-core", "ver": "0.43.0", "Node name for S&R": "CreateVideo"}, "widgets_values": [25, "auto", "sRGB", "none"], "widgets_values_named": {"fps": 25, "bit_depth": "auto", "color_space": "sRGB", "codec": "none"}}, {"id": 8, "type": "SaveVideo", "pos": [1506.932966023875, 298.9898989898991], "size": [280, 170], "flags": {}, "order": 7, "mode": 0, "inputs": [{"localized_name": "vidéo", "name": "video", "type": "VIDEO", "link": 16}, {"localized_name": "préfixe_nom_fichier", "name": "filename_prefix", "type": "STRING", "widget": {"name": "filename_prefix"}, "link": null}, {"localized_name": "format", "name": "format", "type": "COMFY_DYNAMICCOMBO_V3", "widget": {"name": "format"}, "link": null}, {"localized_name": "codec", "name": "format.codec", "type": "COMFY_DYNAMICCOMBO_V3", "widget": {"name": "format.codec"}, "link": null}, {"localized_name": "codec", "name": "codec", "shape": 7, "type": "COMFY_DYNAMICCOMBO_V3", "widget": {"name": "codec"}, "link": null}], "outputs": [{"localized_name": "vidéo", "name": "video", "type": "VIDEO", "links": null}], "title": "SAUVEGARDE VIDEO", "properties": {"cnr_id": "comfy-core", "ver": "0.43.0", "Node name for S&R": "SaveVideo"}, "widgets_values": ["video/ComfyUI", "auto", "auto", "auto"], "widgets_values_named": {"filename_prefix": "video/ComfyUI", "format": "auto", "format.codec": "auto", "codec": "auto"}}], "links": [[10, 1, 0, 3, 1, "STRING"], [11, 2, 0, 3, 5, "STRING"], [12, 3, 0, 5, 0, "STRING"], [13, 3, 1, 6, 0, "STRING"], [14, 3, 2, 4, 0, "IMAGE"], [15, 3, 2, 7, 0, "IMAGE"], [16, 7, 0, 8, 0, "VIDEO"]], "groups": [], "config": {}, "extra": {"ds": {"scale": 0.5445000000000005, "offset": [439.3312734938306, 622.0192305251021]}}, "extensions": {"title": "Kaggle Video LTXV"}, "version": 0.4}}
wd=pathlib.Path('/kaggle/working/ComfyUI/user/default/workflows'); wd.mkdir(parents=True,exist_ok=True)
for name,data in WORKFLOWS.items():
    (wd/name).write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
print('workflows:',list(WORKFLOWS))
print('ETAPE3 FINIE')


In [ ]:
print('ETAPE4: comfy + proxy token-gate securise')
import os, threading, subprocess, sys, time, asyncio, signal
from aiohttp import web, ClientSession
from kaggle_secrets import UserSecretsClient
TOKEN=UserSecretsClient().get_secret('HEPHAISTOS_TOKEN')
if not TOKEN: raise RuntimeError('Kaggle Secret HEPHAISTOS_TOKEN manquant')
comfy=subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-cuda-malloc'],cwd='/kaggle/working/ComfyUI')
_last=[time.time()]
async def proxy(req):
    if req.headers.get('X-Hephaistos-Token') != TOKEN: return web.Response(status=401,text='bad token')
    _last[0]=time.time()
    async with ClientSession() as s:
        body=await req.read() if req.can_read_body else None
        async with s.request(req.method,'http://127.0.0.1:8188'+req.path_qs,data=body,headers={k:v for k,v in req.headers.items() if k.lower() not in ('host','x-hephaistos-token')}) as up:
            return web.Response(status=up.status,body=await up.read(),headers={k:v for k,v in up.headers.items() if k.lower() not in ('content-encoding','content-length','transfer-encoding')})
async def serve():
    app=web.Application(); app.router.add_route('*','/{path:.*}',proxy); runner=web.AppRunner(app); await runner.setup(); await web.TCPSite(runner,'0.0.0.0',8080).start(); print('proxy :8080')
threading.Thread(target=lambda:asyncio.run(serve()),daemon=True).start()
print('ETAPE4 FINIE')


In [ ]:
print('ETAPE5: tunnel nomme')
import subprocess,time,threading
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient(); token=None
for name in ('CF_TUNNEL_TOKEN','CLOUDFLARE_TUNNEL_TOKEN','MAROTTE_TUNNEL_TOKEN'):
    try:
        token=secrets.get_secret(name)
        if token: break
    except Exception: pass
if not token: raise RuntimeError('Secret tunnel manquant')
proc=subprocess.Popen(['/kaggle/working/cloudflared','tunnel','--no-autoupdate','run','--token',token],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
threading.Thread(target=lambda:[print('CF:',line.strip()[:180],flush=True) for line in proc.stdout],daemon=True).start()
time.sleep(20); print('KAGGLE_COMFY_URL=https://comfi.art-qalam.fr'); print('ETAPE5 FINIE')


In [ ]:
print('ETAPE6: watchdog')
import time,os,signal
deadline=time.time()+float(os.environ.get('KERNEL_MAX_HOURS','8'))*3600
while time.time()<deadline:
    if comfy.poll() is not None: raise RuntimeError('ComfyUI mort')
    if proc.poll() is not None: raise RuntimeError('cloudflared mort')
    time.sleep(30)
os.kill(os.getpid(),signal.SIGTERM)
